# 第8章：データの連結・結合・変形（実務寄りテンプレ集）
目的：**こう書いたらこうなる**を最短で体得。まずは一緒に手を動かす。

In [ ]:
# セットアップ
import pandas as pd
from pathlib import Path
DATA = Path.cwd() / "assets"
if not DATA.exists():
    # Colab等で実行する際の相対調整
    DATA = Path("assets")
sales = pd.read_csv(DATA / "sales_sample.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")
sales.head()

## 8.1 concat（縦横に繋ぐ）

In [ ]:
# 例：店舗ごとに分割→縦に結合
tokyo = sales[sales['store']=='Tokyo'].head(5)
osaka = sales[sales['store']=='Osaka'].head(5)
tokyo_osaka = pd.concat([tokyo, osaka], axis=0, ignore_index=True)
tokyo_osaka

In [ ]:
# 例：売上と数量の小計を横結合（列方向）
summary_amount = sales.groupby('store')['amount'].sum().to_frame('sum_amount')
summary_qty = sales.groupby('store')['qty'].sum().to_frame('sum_qty')
store_summary = pd.concat([summary_amount, summary_qty], axis=1)
store_summary

## 8.2 merge/join（キーで結合）

In [ ]:
# 基本：売上明細（product）に製品マスタ（products）を結合
sales_prod = sales.merge(products, on='product', how='left')
sales_prod.head()

In [ ]:
# 顧客属性を付与
sales_all = sales_prod.merge(customers, on='customer_id', how='left')
sales_all[['store','product','category','customer_id','age','gender','pref']].head()

## 8.3 pivot / pivot_table（クロス集計）

In [ ]:
# 店舗×カテゴリの売上合計（pivot_table）
pivot1 = sales_all.pivot_table(index='store', columns='category', values='amount', aggfunc='sum')
pivot1

In [ ]:
# groupby→unstack でも同じ結果を作れる
pivot1_alt = (sales_all.groupby(['store','category'])['amount']
              .sum()
              .unstack('category'))
pivot1_alt

## 8.4 stack / unstack / melt（縦横変形の往復）

In [ ]:
# 横持ち → 縦持ち：unstack後→stackで戻す
wide = pivot1_alt
long = wide.stack().reset_index(name='amount_sum')
round_trip = long.pivot_table(index='store', columns='category', values='amount_sum', aggfunc='sum')
(wide.equals(round_trip), wide.shape, round_trip.shape)

In [ ]:
# 任意の列をmeltで縦長化（tidyデータ化）
tmp = sales_all[['store','category','qty','amount']].head(10)
melted = tmp.melt(id_vars=['store','category'], value_vars=['qty','amount'], var_name='metric', value_name='value')
melted

## 8.5 小課題：CRM×購買明細の結合→カテゴリ×月のクロス集計
- 手順ヒント：`merge`→`to_datetime`→`dt.to_period('M')`→`pivot_table`

In [ ]:
# === ここから自分で ===
df = sales.merge(products, on='product', how='left').merge(customers, on='customer_id', how='left')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['month'] = df['date'].dt.to_period('M')
xtab = df.pivot_table(index=['pref'], columns=['category','month'], values='amount', aggfunc='sum')
xtab.head(8)